# How Data Flows Through an RNN Cell and an LSTM Cell
### A step-by-step toy example you can check with a calculator

This notebook uses **only toy data**: 2 short sentences, a 6-word vocabulary, 2 numbers per word, and 2 memory units. Every weight is a small, fixed, hand-picked number (not random), so every value printed below can be recomputed by hand.

The plan:

| Part | What we do |
|---|---|
| 1 | Turn 2 sentences into a 3D tensor `(batch, timesteps, features)` |
| 2 | Push one sentence through an **RNN cell**, one time step at a time |
| 3 | Do the same for the whole batch, and see what the layer returns |
| 4 | Push the same sentence through an **LSTM cell**, gate by gate |
| 5 | Turn the final memory into a prediction |
| 6 | Check every hand calculation against real Keras layers |
| 7 | RNN vs LSTM side by side, and practice questions |

In [1]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)

def vec(v):
    # print a vector with 4 decimals
    return "[" + ", ".join(f"{x:.4f}" for x in np.atleast_1d(v)) + "]"

---
# Part 1: From sentences to a 3D tensor

### Step 1.1: The vocabulary and the sentences

Each word gets an integer ID. ID **0 is kept for padding**, so real words start at 1.

In [2]:
vocab = {"<pad>": 0, "movie": 1, "was": 2, "not": 3, "good": 4, "great": 5, "boring": 6}
sentences = ["movie was great", "movie was not good"]

ids = [[vocab[w] for w in s.split()] for s in sentences]
for s, i in zip(sentences, ids):
    print(f"{s!r:22} -> {i}   (length {len(i)})")

'movie was great'      -> [1, 2, 5]   (length 3)
'movie was not good'   -> [1, 2, 3, 4]   (length 4)


### Step 1.2: Padding, so every sentence has the same length

The sentences have lengths 3 and 4. A network needs a rectangle, so we pad to `maxlen = 4` by adding 0s **at the front** (pre-padding, the Keras default). Now the real words sit at the end, right before the RNN gives its final answer.

In [3]:
maxlen = 4
padded = np.array([[0] * (maxlen - len(s)) + s for s in ids])   # same as pad_sequences(ids, maxlen=4)
print(padded)
print("\nshape:", padded.shape, "-> (batch = 2 sentences, timesteps = 4 words)")

[[0 1 2 5]
 [1 2 3 4]]

shape: (2, 4) -> (batch = 2 sentences, timesteps = 4 words)


Read this matrix like a timetable:

| | t = 0 | t = 1 | t = 2 | t = 3 |
|---|---|---|---|---|
| sentence 0 | `<pad>` | movie | was | great |
| sentence 1 | movie | was | not | good |

**Rows = sentences (batch). Columns = time steps.** An RNN walks across the columns, left to right.

### Step 1.3: Embedding, each word ID becomes a small vector

An embedding is a lookup table: row `k` holds the vector for word ID `k`. In real models these numbers are learned. Here I've hand-picked them so they're easy to read: the **first number** roughly means "how positive is this word", the second is a bit of extra meaning.

In [4]:
E = np.array([
    [ 0.0,  0.0],   # 0 <pad>
    [ 0.1,  0.2],   # 1 movie
    [ 0.0,  0.1],   # 2 was
    [-0.5,  0.3],   # 3 not
    [ 0.8,  0.1],   # 4 good
    [ 0.9,  0.2],   # 5 great
    [-0.7, -0.2],   # 6 boring
])
print(pd.DataFrame(E, index=list(vocab.keys()), columns=["feature_1", "feature_2"]))

        feature_1  feature_2
<pad>         0.0        0.0
movie         0.1        0.2
was           0.0        0.1
not          -0.5        0.3
good          0.8        0.1
great         0.9        0.2
boring       -0.7       -0.2


In [5]:
X = E[padded]            # lookup: replace every ID with its row of E
print("X shape:", X.shape, "-> (batch = 2, timesteps = 4, features = 2)\n")
for b, s in enumerate(sentences):
    print(f"Sentence {b}: {s!r}")
    for t in range(maxlen):
        word = list(vocab.keys())[padded[b, t]]
        print(f"   t={t}  {word:7} id={padded[b, t]}  ->  x_t = {vec(X[b, t])}")

X shape: (2, 4, 2) -> (batch = 2, timesteps = 4, features = 2)

Sentence 0: 'movie was great'
   t=0  <pad>   id=0  ->  x_t = [0.0000, 0.0000]
   t=1  movie   id=1  ->  x_t = [0.1000, 0.2000]
   t=2  was     id=2  ->  x_t = [0.0000, 0.1000]
   t=3  great   id=5  ->  x_t = [0.9000, 0.2000]
Sentence 1: 'movie was not good'
   t=0  movie   id=1  ->  x_t = [0.1000, 0.2000]
   t=1  was     id=2  ->  x_t = [0.0000, 0.1000]
   t=2  not     id=3  ->  x_t = [-0.5000, 0.3000]
   t=3  good    id=4  ->  x_t = [0.8000, 0.1000]


This 3D shape `(batch, timesteps, features)` is **exactly what every RNN and LSTM layer expects as input**. From here on, at each time step the cell receives one small vector `x_t` of 2 numbers.

---
# Part 2: Inside the RNN cell, one time step at a time

### The idea

The RNN cell carries a **memory vector `h`** (here 2 numbers, because `units = 2`). It starts as `[0, 0]`. At every time step it mixes the **current word** with the **previous memory**:

$$h_t = \tanh(\;x_t \cdot W_x \;+\; h_{t-1} \cdot W_h \;+\; b\;)$$

```
             x_t (current word)
                  |
                  v
  h_(t-1) ---> [ RNN cell ] ---> h_t  ---> (next time step, and output)
 (old memory)                   (new memory)
```

### The weights (fixed, so you can check by hand)

| Name | Keras name | Shape | Job |
|---|---|---|---|
| `Wx` | `kernel` | `(features=2, units=2)` | how the current word affects memory |
| `Wh` | `recurrent_kernel` | `(units=2, units=2)` | how old memory affects new memory |
| `b` | `bias` | `(units=2,)` | a constant nudge |

The **same weights are reused at every time step**. That's what "recurrent" means.

In [6]:
Wx = np.array([[0.5, -0.3],
               [0.2,  0.4]])
Wh = np.array([[0.1,  0.2],
               [-0.1, 0.3]])
b  = np.array([0.0, 0.1])

print("Wx =\n", Wx, "\nWh =\n", Wh, "\nb =", b)
print("\nRNN parameters = units x (features + units + 1) = 2 x (2 + 2 + 1) =", Wx.size + Wh.size + b.size)

Wx =
 [[ 0.5 -0.3]
 [ 0.2  0.4]] 
Wh =
 [[ 0.1  0.2]
 [-0.1  0.3]] 
b = [0.  0.1]

RNN parameters = units x (features + units + 1) = 2 x (2 + 2 + 1) = 10


### Step 2.1: Sentence 1, "movie was not good", fully expanded

I'll print every multiplication for each step so nothing is hidden. Pick any line and verify it on a calculator.

In [ ]:
def show_matvec(v, W, name_v, name_W):
    # prints v @ W with the arithmetic written out
    out = v @ W
    for j in range(W.shape[1]):
        terms = " + ".join(f"({v[k]:.4f} x {W[k, j]:.1f})" for k in range(len(v)))
        print(f"      [{name_v} . {name_W}]_{j} = {terms} = {out[j]:.4f}")
    return out

sent = 1
x_seq = X[sent]
h = np.zeros(2)
rnn_h_history = []

for t in range(maxlen):
    word = list(vocab.keys())[padded[sent, t]]
    print(f"=============== t = {t}   word = '{word}' ===============")
    print(f"   x_t     = {vec(x_seq[t])}")
    print(f"   h_(t-1) = {vec(h)}")
    a = show_matvec(x_seq[t], Wx, "x_t", "Wx")
    r = show_matvec(h, Wh, "h_(t-1)", "Wh")
    total = a + r + b
    print(f"   sum     = {vec(a)} + {vec(r)} + {vec(b)} = {vec(total)}")
    h = np.tanh(total)
    print(f"   h_t     = tanh(sum) = {vec(h)}\n")
    rnn_h_history.append(h)

print("Final memory after reading the whole sentence:", vec(h))

What to notice:

**At t = 0, `h_(t-1)` is `[0, 0]`**, so only the word matters. From t = 1 onwards, the old memory is mixed in. That's how "movie" still has a small influence when the cell reads "good".

**`tanh` keeps every value between -1 and 1**, so the memory can never blow up.

**The final `h` is a summary of the whole sentence**, squeezed into 2 numbers.

### Step 2.2: The same thing as a table

In [ ]:
pd.DataFrame(
    [{"t": t, "word": list(vocab.keys())[padded[sent, t]],
      "x_t": vec(x_seq[t]), "h_t": vec(rnn_h_history[t])} for t in range(maxlen)]
).set_index("t")

### Step 2.3: What does padding do?

Sentence 0 starts with `<pad>`. Its vector is `[0, 0]` and the starting memory is `[0, 0]`, so at t = 0:

$$h_0 = \tanh(0 + 0 + b) = \tanh([0.0,\ 0.1]) = [0.0,\ 0.0997]$$

The memory is **not** zero after a pad step, because of the bias. The cell "reads" the padding like a real word. (In Keras, `Embedding(..., mask_zero=True)` tells the RNN to skip pad steps entirely.)

---
# Part 3: The whole batch at once, and what the layer returns

### Step 3.1: Time steps are sequential, sentences are parallel

At time step `t` the layer takes **column t of every sentence** and processes them together with one matrix multiplication:

```
x_t      : (batch=2, features=2)
h_(t-1)  : (batch=2, units=2)
x_t @ Wx : (2,2) @ (2,2) -> (2,2)
h @ Wh   : (2,2) @ (2,2) -> (2,2)
h_t      : (2,2)   one memory vector per sentence
```

In [ ]:
H = np.zeros((2, 2))
all_steps = []
for t in range(maxlen):
    x_t = X[:, t, :]                        # word t of BOTH sentences
    H = np.tanh(x_t @ Wx + H @ Wh + b)
    all_steps.append(H.copy())
    print(f"t={t}: x_t {x_t.shape} -> h_t {H.shape}\n{H}\n")

rnn_all = np.stack(all_steps, axis=1)       # (batch, timesteps, units)

### Step 3.2: `return_sequences=True` vs `False`

The layer calculated `h_t` at every step. You choose what it hands to the next layer:

| Setting | What comes out | Shape here | Next layer is usually |
|---|---|---|---|
| `return_sequences=True` | `h_t` for **every** step | `(2, 4, 2)` | another RNN / LSTM |
| `return_sequences=False` (default) | only the **last** `h_t` | `(2, 2)` | Dense (classification) |

In [ ]:
print("return_sequences=True  -> shape", rnn_all.shape)
print(rnn_all)
print("\nreturn_sequences=False -> shape", rnn_all[:, -1, :].shape)
print(rnn_all[:, -1, :])

---
# Part 4: Inside the LSTM cell, gate by gate

### Why another cell?

In the RNN, the whole memory is rewritten through `tanh` at every step. Over a long sentence, early words fade away. An LSTM fixes this by keeping **two** memories:

| Memory | Name | Role |
|---|---|---|
| `c` | cell state | long-term memory, the "conveyor belt" |
| `h` | hidden state | short-term memory, and the **output** of the cell |

```
  c_(t-1) ----[ x f ]------[ + ]--------------------> c_t
                              ^                 |
                           [ i x g ]          tanh
                                                |
  h_(t-1) --+                                 [ x o ] ---> h_t
            +--> forget f, input i, candidate g, output o
  x_t ------+
```

### The four calculations at every step

Each one looks at the same two things, `x_t` and `h_(t-1)`, but with its **own weights**:

| Symbol | Name | Activation | Range | Question it answers |
|---|---|---|---|---|
| `f` | forget gate | sigmoid | 0 to 1 | how much of the old `c` should I keep? |
| `i` | input gate | sigmoid | 0 to 1 | how much new information should I write? |
| `g` | candidate | tanh | -1 to 1 | what is the new information? |
| `o` | output gate | sigmoid | 0 to 1 | how much of `c` should I show as `h`? |

Then the two memory updates:

$$c_t = f \odot c_{t-1} \;+\; i \odot g \qquad\qquad h_t = o \odot \tanh(c_t)$$

($\odot$ means multiply element by element.)

In [ ]:
# Each gate has its own W (for x_t), U (for h_(t-1)) and bias
Wi = np.array([[0.4, 0.1], [0.2, 0.3]]);   Ui = np.array([[0.1, 0.0], [0.0, 0.1]]);   bi = np.array([0.0, 0.0])
Wf = np.array([[0.3, 0.2], [0.1, 0.4]]);   Uf = np.array([[0.2, 0.1], [0.1, 0.2]]);   bf = np.array([1.0, 1.0])
Wg = np.array([[0.6, -0.4], [0.2, 0.5]]);  Ug = np.array([[0.1, 0.1], [-0.1, 0.1]]);  bg = np.array([0.0, 0.0])
Wo = np.array([[0.3, 0.3], [0.2, -0.2]]);  Uo = np.array([[0.1, 0.0], [0.0, 0.1]]);   bo = np.array([0.0, 0.0])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

n_params = sum(w.size for w in [Wi, Ui, bi, Wf, Uf, bf, Wg, Ug, bg, Wo, Uo, bo])
print("LSTM parameters = 4 x units x (features + units + 1) = 4 x 2 x (2 + 2 + 1) =", n_params)
print("That's 4x the RNN, because there are 4 sets of weights (f, i, g, o).")

The forget bias `bf` is set to **1**, like Keras does by default. `sigmoid(1) = 0.73`, so a fresh LSTM starts out leaning towards **remembering**.

### Step 4.1: Sentence 1, "movie was not good", every gate printed

In [ ]:
h = np.zeros(2)
c = np.zeros(2)
lstm_rows = []

for t in range(maxlen):
    x_t = X[sent, t]
    word = list(vocab.keys())[padded[sent, t]]
    print(f"=============== t = {t}   word = '{word}' ===============")
    print(f"   inputs : x_t = {vec(x_t)}   h_(t-1) = {vec(h)}   c_(t-1) = {vec(c)}")

    zf = x_t @ Wf + h @ Uf + bf;  f = sigmoid(zf)
    zi = x_t @ Wi + h @ Ui + bi;  i = sigmoid(zi)
    zg = x_t @ Wg + h @ Ug + bg;  g = np.tanh(zg)
    zo = x_t @ Wo + h @ Uo + bo;  o = sigmoid(zo)
    print(f"   forget  f = sigmoid({vec(zf)}) = {vec(f)}")
    print(f"   input   i = sigmoid({vec(zi)}) = {vec(i)}")
    print(f"   candid. g = tanh   ({vec(zg)}) = {vec(g)}")
    print(f"   output  o = sigmoid({vec(zo)}) = {vec(o)}")

    keep  = f * c
    write = i * g
    c = keep + write
    print(f"   c_t = f*c_(t-1) + i*g = {vec(keep)} + {vec(write)} = {vec(c)}")
    h = o * np.tanh(c)
    print(f"   h_t = o*tanh(c_t) = {vec(o)} * {vec(np.tanh(c))} = {vec(h)}\n")

    lstm_rows.append({"t": t, "word": word, "f": vec(f), "i": vec(i), "g": vec(g),
                      "o": vec(o), "c_t": vec(c), "h_t": vec(h)})

print("Final h (the LSTM's output):", vec(h))
print("Final c (long-term memory)  :", vec(c))

### Step 4.2: Everything in one table

In [ ]:
pd.DataFrame(lstm_rows).set_index("t")

How to read the table with students:

**Follow the `c_t` column.** It changes by *keep a fraction of the old value, then add something new*. No `tanh` squashes it on the way, which is why information can survive many steps. This is the "conveyor belt".

**At "not", the candidate `g` turns negative** in the first unit (because "not" has a negative first feature). The cell writes that into `c`.

**At "good", the first unit of `c` gets a big positive write (about +0.26)**, which flips it from -0.075 back to +0.205. The final memory carries the effect of both words, and that's what training later uses to tell "not good" apart from "good".

**`h_t` is always smaller in size than `c_t`**, because it's `c_t` squashed by `tanh` and then scaled by the output gate. `c` is the storage, `h` is what the cell chooses to show.

---
# Part 5: From the final memory to a prediction

For sentiment classification we keep only the **last** `h` (`return_sequences=False`) and pass it to `Dense(1, activation='sigmoid')`:

$$\hat{y} = \sigma(h_T \cdot W_y + b_y)$$

`Dense` doesn't know anything about sequences. It just sees 2 numbers per sentence and turns them into one probability.

In [ ]:
Wy = np.array([[2.0], [1.0]])
by = np.array([0.0])

rnn_final = rnn_all[:, -1, :]            # from Part 3
lstm_final = []
for s in range(2):                       # run LSTM on both sentences
    h, c = np.zeros(2), np.zeros(2)
    for t in range(maxlen):
        x_t = X[s, t]
        f = sigmoid(x_t @ Wf + h @ Uf + bf); i = sigmoid(x_t @ Wi + h @ Ui + bi)
        g = np.tanh(x_t @ Wg + h @ Ug + bg); o = sigmoid(x_t @ Wo + h @ Uo + bo)
        c = f * c + i * g
        h = o * np.tanh(c)
    lstm_final.append(h)
lstm_final = np.array(lstm_final)

for name, H_last in [("RNN ", rnn_final), ("LSTM", lstm_final)]:
    z = H_last @ Wy + by
    p = sigmoid(z)
    for s in range(2):
        print(f"{name} | {sentences[s]!r:22} h_T = {vec(H_last[s])}  ->  z = {z[s,0]:.4f}  ->  P(positive) = {p[s,0]:.4f}")

These probabilities mean nothing yet, because the weights were hand-picked, not trained. Training is the process of adjusting `E`, `Wx`, `Wh`, the gate weights and `Wy` until these probabilities match the true labels. The **data flow** you just traced stays exactly the same before and after training.

---
# Part 6: Proof, the same numbers from real Keras layers

We load our hand-picked weights into Keras layers and compare. Keras stores the LSTM's 4 gates side by side in one matrix, in the order **i, f, g (called c), o**.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM

emb = Embedding(input_dim=7, output_dim=2)
emb.build((None, maxlen)); emb.set_weights([E])
X_keras = emb(padded).numpy()
print("Embedding matches E lookup:", np.allclose(X_keras, X))

rnn_layer = SimpleRNN(2, return_sequences=True)
rnn_layer.build((None, maxlen, 2)); rnn_layer.set_weights([Wx, Wh, b])
print("SimpleRNN matches my loop :", np.allclose(rnn_layer(X_keras).numpy(), rnn_all, atol=1e-6))

lstm_layer = LSTM(2)
lstm_layer.build((None, maxlen, 2))
lstm_layer.set_weights([np.concatenate([Wi, Wf, Wg, Wo], axis=1),     # kernel           (2, 8)
                        np.concatenate([Ui, Uf, Ug, Uo], axis=1),     # recurrent_kernel (2, 8)
                        np.concatenate([bi, bf, bg, bo])])            # bias             (8,)
print("LSTM matches my loop      :", np.allclose(lstm_layer(X_keras).numpy(), lstm_final, atol=1e-6))

---
# Part 7: RNN vs LSTM side by side

| | RNN cell | LSTM cell |
|---|---|---|
| Takes in at each step | `x_t`, `h_(t-1)` | `x_t`, `h_(t-1)`, `c_(t-1)` |
| Gives out at each step | `h_t` | `h_t`, `c_t` |
| Formula | `h_t = tanh(x_t Wx + h_(t-1) Wh + b)` | 4 gates, then `c_t = f*c + i*g`, `h_t = o*tanh(c_t)` |
| Layer output | `h_t` (last, or all) | `h_t` (last, or all). `c` stays inside |
| Parameters | `u(d + u + 1)` = 10 here | `4u(d + u + 1)` = 40 here |
| Long sentences | early words fade | cell state carries them further |

`u` = units, `d` = features per time step.

### Shapes to remember (our toy)

```
IDs                    (2, 4)
after Embedding        (2, 4, 2)     batch, timesteps, features
RNN / LSTM, seq=True   (2, 4, 2)     batch, timesteps, units
RNN / LSTM, seq=False  (2, 2)        batch, units
Dense(1)               (2, 1)        one probability per sentence
```

### Practice questions

1. Using the weights above, compute the RNN's `h_0` for sentence 1 ("movie") by hand.
2. Why is `h_0` for sentence 0 not `[0, 0]`, even though the word is `<pad>`?
3. If `units = 3` and the embedding size is 5, how many parameters do a SimpleRNN and an LSTM have?
4. If you stack two LSTMs, which one needs `return_sequences=True`, and why?
5. In the LSTM table, which gate would you need to push close to 0 for the cell to "forget" everything before the word "not"?

*Answers: (1) tanh([0.1x0.5 + 0.2x0.2, 0.1x-0.3 + 0.2x0.4] + [0, 0.1]) = tanh([0.09, 0.15]) = [0.0898, 0.1489]. (2) The bias b is still added, and tanh(b) is not zero. (3) RNN 3 x (5+3+1) = 27, LSTM 4 x 27 = 108. (4) The first one, because the second LSTM needs a full sequence as input. (5) The forget gate f at that step.*